# Lab: Learning to Move Through a Trocar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LxViRaL-teaching/medical-robotics-2026/blob/main/week-7/robmed_rl_lab.ipynb)

We go back to the S2RK surgical robot of week 4, with its instrument through a trocar port. The task:

* the tool tip must go round a **circle** of radius 15 mm, anticlockwise, at **1 cm/s**;
* the instrument must **not touch the port**: it may pivot about the trocar point (the RCM) and slide, but not push sideways.

In week 4 you kept the RCM with a model. This time the robot has to **learn** how to tilt the shaft while the tip moves, by trial and error, with **Q-learning**: the table method from the lecture, the one we filled in on the board.

Everything around the learning is ready in `circle_utils.py`: the robot, the states, the actions, the training loop. **You write three things:**

1. the **reward**: what counts as doing well;
2. how the agent **chooses an action**: ε-greedy;
3. the **Q-learning update**: how the table learns from each step.

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybullet"], check=True)
    if not os.path.exists("medical-robotics-2026"):
        subprocess.run(["git", "clone", "-q", "https://github.com/LxViRaL-teaching/medical-robotics-2026.git"], check=True)
    os.chdir("medical-robotics-2026/week-7")

import numpy as np
import circle_utils as cl

REAL_TIME = True     # runs on the robot go at real speed, so you can watch them in the 3D window

## 1. The scene

The robot, the trocar port with its force sensor, and the circle (drawn in blue). The circle lies in the plane perpendicular to the instrument at the start pose.

In [2]:
lab = cl.CircleLab(real_time=REAL_TIME)
print("circle radius:", cl.RADIUS * 1000, "mm,  speed:", cl.SPEED * 1000, "mm/s")

circle radius: 15.0 mm,  speed: 10.0 mm/s


## 2. Following the circle without tilting

First, a hand-written controller: the tip moves along the circle at 1 cm/s and the shaft **does not tilt**. We measure the **offset** of the shaft at the port (how far the shaft passes from the trocar point) and the **force on the port**.

In [3]:
log = lab.run_without_tilt(duration=4.0)
cl.plot_series(log["time"], {"shaft offset at the port [mm]": log["offset_mm"]}, ylabel="[mm]",
               title="Without tilting: offset at the port (the shaft touches the ring at 7.5 mm)")
cl.plot_series(log["time"], {"force on the port [N]": log["force"]}, ylabel="[N]",
               title="Without tilting: force on the port")

The tip follows the circle, but the shaft only **translates**: it slides sideways at the trocar until it hits the ring.
To stay on the RCM the shaft must **tilt** while the tip moves, by just the right amount. The agent will learn it.

## 3. The learning problem

**Actions.** Every 0.1 s the agent chooses a tip velocity (each of its three components $-1$, $0$ or $+1$ × 1 cm/s) and a tilt of the shaft ($-1$, $0$ or $+1$ × 0.07 rad/s): $3^4 = 81$ actions.

**States.** The table needs a finite number of states, so the measurements are cut into bins:

| Measurement | Bins |
|---|---|
| where the tip is around the circle | 4 sectors |
| distance to the circle | 4 bins |
| offset of the shaft at the port | 3 bins |
| height of the tip above the plane of the circle | 3 bins |

$4 \times 4 \times 3 \times 3 = 144$ states. The Q-table is $144 \times 81$.

In [4]:
print(lab.n_states, "states,", lab.n_actions, "actions,", lab.n_states * lab.n_actions, "table entries")

144 states, 81 actions, 11664 table entries


## 4. The reward

After every action, the reward function receives what happened:

| Name | Meaning | Unit |
|---|---|---|
| `tangential_speed` | speed of the tip along the circle (positive = anticlockwise) | m/s |
| `radial_error` | distance from the centre of the circle minus the radius (positive = outside) | m |
| `height` | distance of the tip from the plane of the circle | m |
| `shaft_offset` | distance from the trocar point to the shaft (signed) | m |
| `port_penalty` | `True` or `False`: whether to penalise the shaft offset | |

and these constants: `cl.SPEED` = 0.01 m/s (the desired speed) and `cl.CONTACT_OFFSET` = 0.0075 m (beyond it, the shaft touches the port).

Write the reward

$$r = -\frac{|v_t - v^*|}{v^*} - \left(\frac{e_r}{3\,\text{mm}}\right)^2 - \left(\frac{h}{3\,\text{mm}}\right)^2 \;\underbrace{-\left(\frac{d}{7.5\,\text{mm}}\right)^2 - 10\,[\,|d| > 7.5\,\text{mm}\,]}_{\text{only if port\_penalty}}$$

with $v_t$ the tangential speed, $v^*$ = `cl.SPEED`, $e_r$ the radial error, $h$ the height and $d$ the shaft offset. The last term, $-10\,[\dots]$, applies only when the shaft touches the port.

In [5]:
def reward(tangential_speed, radial_error, height, shaft_offset, port_penalty):
    speed_term = -abs(tangential_speed - cl.SPEED) / cl.SPEED
    
    #TODO: add circle and height terms to the reward
    circle_term = - (radial_error / 3)**2
    height_term = - (height / 3)**2
    total = speed_term + circle_term + height_term
    if port_penalty:
        port_term = -(shaft_offset / cl.CONTACT_OFFSET) ** 2
        if abs(shaft_offset) > cl.CONTACT_OFFSET:
        #TODO Penalize if collided
            total = total + port_term - 10
    return total


print("on the circle, at the right speed, centred:", reward(0.01, 0.0, 0.0, 0.0, True))      # expected 0
print("standing still:", reward(0.0, 0.0, 0.0, 0.0, True))                                 # expected -1
print("touching the port:", reward(0.01, 0.0, 0.0, 0.008, True))                          # expect big penalty

on the circle, at the right speed, centred: -0.0
standing still: -1.0
touching the port: -11.137777777777778


## 5. Choosing actions: ε-greedy

At every step the agent must choose an action. Two ways:

* **exploit**: take the best action according to the table so far, the one with the highest $Q(s, a)$ in the current state;
* **explore**: take a random action, to try things the table does not know about yet.

**ε-greedy** mixes them: with probability ε explore, otherwise exploit. During training ε starts at 0.3 (a lot of exploring, when the table knows nothing) and goes down to 0.02.

`rng` is a numpy random generator: `rng.random()` gives a number between 0 and 1, and `rng.integers(n)` an integer from 0 to n−1. The actions are numbered from 0 to `lab.n_actions - 1`.

In [13]:
def pick_best_action(Q, state):
    # The action with the highest Q in this state
    return int(np.argmax(Q[state]))


def pick_random_action(rng):
    # Any action, all equally likely
    return int(rng.integers(lab.n_actions))


def choose_action(Q, state, epsilon, rng):
    #TODO: epsilon-greedy: explore with probability epsilon, otherwise exploit
    if rng.random() <= epsilon: return pick_random_action(rng)
    else: return pick_best_action(Q, state)



rng_test = np.random.default_rng(0)
Q_test = np.zeros((1, lab.n_actions))
Q_test[0, 7] = 1.0                                          # in state 0, action 7 is the best
picks = [choose_action(Q_test, 0, 0.3, rng_test) for _ in range(1000)]
print("best action:", pick_best_action(Q_test, 0), "(expected 7)")
print(f"with epsilon = 0.3, action 7 was chosen {picks.count(7) / 10:.0f}% of the time (expected about 70%)")

best action: 7 (expected 7)
with epsilon = 0.3, action 7 was chosen 72% of the time (expected about 70%)


## 6. The Q-learning update

After every step, the training loop calls `q_update(Q, state, action, r, next_state, episode_ended)`. It must apply the rule from the lecture:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s,a) \right]$$

When the episode has ended there is no future: the target is just $r$.

`Q` is a numpy array with one row per state and one column per action: `Q[state, action]` is one entry, `Q[next_state]` is the whole row of the next state.

In [35]:
alpha = 0.2      # learning rate
gamma = 0.9      # discount factor


def q_update(Q, state, action, r, next_state, episode_ended):
    if episode_ended:
        target = r
    else:
        target = r + gamma * np.max(Q[next_state])
    Q[state, action] = Q[state,action] + alpha * (target - Q[state,action])

A quick check on a table with two states, before training on 11,664 entries:

In [36]:
Q_test = np.zeros((2, 2))
Q_test[1] = [1.0, 3.0]                                   # the row of state 1
q_update(Q_test, 0, 0, -1.0, 1, episode_ended=False)
q_update(Q_test, 0, 1, 10.0, 1, episode_ended=True)
expected = [alpha * (-1.0 + gamma * 3.0), alpha * 10.0]
print("Q_test[0] =", Q_test[0], " expected", expected, " ->", "OK" if np.allclose(Q_test[0], expected) else "WRONG")

Q_test[0] = [0.34 2.  ]  expected [0.3400000000000001, 2.0]  -> OK


## 7. Training

`lab.train` runs the episodes on a copy of the robot **without physics** (only kinematics), which is fast: 1000 episodes of 30 s each take about a minute.
At every step it calls your `choose_action`, your `reward` and your `q_update`.

In [37]:
Q_penalty, returns_penalty, visits_penalty = lab.train(reward, q_update, choose_action, port_penalty=True)

cl.plot_series(list(range(1, len(returns_penalty) + 1)), {"return": returns_penalty},
               xlabel="episode", ylabel="return", title="Learning curve (with port penalty)")
print("table entries visited at least once:", np.count_nonzero(visits_penalty), "of", visits_penalty.size)
print("table entries visited fewer than 10 times:", np.count_nonzero(visits_penalty < 10))

episode 100: mean return of the last 100 = -1045
episode 200: mean return of the last 100 = -388
episode 300: mean return of the last 100 = -230
episode 400: mean return of the last 100 = -208
episode 500: mean return of the last 100 = -189
episode 600: mean return of the last 100 = -149
episode 700: mean return of the last 100 = -126
episode 800: mean return of the last 100 = -97
episode 900: mean return of the last 100 = -85
episode 1000: mean return of the last 100 = -76
training time: 192 s


table entries visited at least once: 11664 of 11664
table entries visited fewer than 10 times: 6677


## 8. The learned policy on the full simulation

The table was learned on the robot without physics, where the port is only a number. Now we use it on the full simulation, with motors, gravity and the port ring. No more exploring: at every decision the robot takes `pick_best_action`.

In [38]:
run_penalty = lab.run_policy(Q_penalty, pick_best_action, duration=30.0)
cl.plot_paths({"with port penalty": (run_penalty["x1"], run_penalty["x2"])}, title="Tip path, full simulation")
cl.plot_series(run_penalty["time"], {"shaft offset [mm]": run_penalty["offset_mm"]}, ylabel="[mm]",
               title="Shaft offset at the port (contact at 7.5 mm)")

largest force on the port: 0.0 N


## 9. What if the reward does not mention the port?

Same everything, but `port_penalty=False`: the reward only asks for the speed and for staying on the circle.

In [39]:
Q_free, returns_free, visits_free = lab.train(reward, q_update, choose_action, port_penalty=False)
run_free = lab.run_policy(Q_free, pick_best_action, duration=30.0)

cl.plot_series(run_penalty["time"], {"with port penalty": run_penalty["offset_mm"], "without": run_free["offset_mm"]},
               ylabel="[mm]", title="Shaft offset at the port (contact at 7.5 mm)")
cl.plot_series(run_penalty["time"], {"with port penalty": run_penalty["force"], "without": run_free["force"]},
               ylabel="[N]", title="Force on the port")
cl.plot_paths({"with port penalty": (run_penalty["x1"], run_penalty["x2"]),
               "without": (run_free["x1"], run_free["x2"])}, title="Tip path, full simulation")

episode 100: mean return of the last 100 = -217
episode 200: mean return of the last 100 = -173
episode 300: mean return of the last 100 = -157
episode 400: mean return of the last 100 = -139
episode 500: mean return of the last 100 = -128
episode 600: mean return of the last 100 = -114
episode 700: mean return of the last 100 = -103
episode 800: mean return of the last 100 = -92
episode 900: mean return of the last 100 = -92
episode 1000: mean return of the last 100 = -74
training time: 192 s
largest force on the port: 38.2 N


Try other seeds: the result changes from seed to seed. Why?

In [40]:
Q_free_1, _, _ = lab.train(reward, q_update, choose_action, port_penalty=False, seed=1)
run_free_1 = lab.run_policy(Q_free_1, pick_best_action, duration=30.0)

episode 100: mean return of the last 100 = -217
episode 200: mean return of the last 100 = -176
episode 300: mean return of the last 100 = -152
episode 400: mean return of the last 100 = -141
episode 500: mean return of the last 100 = -130
episode 600: mean return of the last 100 = -113
episode 700: mean return of the last 100 = -107
episode 800: mean return of the last 100 = -96
episode 900: mean return of the last 100 = -89
episode 1000: mean return of the last 100 = -87
training time: 185 s
largest force on the port: 305.9 N
